# Boston Housing — Exploratory Data Analysis

This notebook:
- Loads the processed Boston housing data from `data/boston.csv`.
- Produces **richer, visible results** (tables + plots) and **persists artifacts** to disk.
- Shows how to export the notebook to **Quarto `.qmd`** and render to HTML.

> If you need to regenerate `data/boston.csv` from the raw text file, run:
>
> `python src/00_rawdata_process.py`

## 1) Set Up Environment (Python + Jupyter + Quarto)

If you haven’t installed Python packages yet (in this repo’s `.venv`), you can run:

```bash
pip install -U pandas numpy matplotlib seaborn scipy scikit-learn pyyaml
```

If you want Quarto export/rendering, install Quarto and verify:

```bash
quarto --version
```

In [1]:
# Imports + versions (reproducibility)
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import scipy
from scipy import stats

import sklearn

print('python:', os.sys.version.split()[0])
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('matplotlib:', matplotlib.__version__)
print('seaborn:', sns.__version__)
print('scipy:', scipy.__version__)
print('sklearn:', sklearn.__version__)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120


python: 3.11.14
numpy: 2.4.2
pandas: 3.0.1
matplotlib: 3.10.8
seaborn: 0.13.2
scipy: 1.17.1
sklearn: 1.8.0


## 2) Reproducible Analysis Cell (Load + Basic Validation)

This section loads `data/boston.csv`, checks expected columns, and creates an output folder for persisted results.

In [ ]:
# Parameters (edit here and re-run)
np.random.seed(0)

DATA_PATH = Path('data/boston.csv')
OUTPUT_DIR = Path('outputs/boston_eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    'CRIM','ZN','INDUS','CHAS','NOX','RM','AGE','DIS','RAD',
    'TAX','PTRATIO','B','LSTAT','MEDV'
]

df = pd.read_csv(DATA_PATH)

print('Loaded:', DATA_PATH)
print('Shape:', df.shape)

missing_total = int(df.isna().sum().sum())
duplicates = int(df.duplicated().sum())

print('Missing total:', missing_total)
print('Duplicate rows:', duplicates)

missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
if len(missing_cols) > 0:
    display(missing_cols)

# Column validation
missing_expected = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]

print('Missing expected columns:', missing_expected)
print('Extra columns:', extra_cols)

display(df.head())


## 3) Show More Results: Summary Tables

We’ll compute:
- `describe()` table for all numeric columns
- value counts for `CHAS`
- a few group-by summaries
- top/bottom `MEDV` rows

In [ ]:
# Numeric summary table
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[numeric_cols].describe().T

display(desc)

# CHAS distribution
if 'CHAS' in df.columns:
    print('CHAS value counts:')
    display(df['CHAS'].value_counts(dropna=False))

# GroupBy example: MEDV by CHAS
if 'CHAS' in df.columns and 'MEDV' in df.columns:
    medv_by_chas = df.groupby('CHAS', dropna=False)['MEDV'].agg(['count','mean','median','std','min','max'])
    print('MEDV grouped by CHAS:')
    display(medv_by_chas)

# Top/bottom MEDV rows
if 'MEDV' in df.columns:
    print('Top 10 MEDV:')
    display(df.sort_values('MEDV', ascending=False).head(10))
    print('Bottom 10 MEDV:')
    display(df.sort_values('MEDV', ascending=True).head(10))


## 4) Show More Results: Correlations and Collinearity

We compute both **Pearson** (linear) and **Spearman** (rank-based) correlations with `MEDV`, plus the strongest predictor–predictor correlations (for multicollinearity awareness).

In [ ]:
# Correlations with MEDV
if 'MEDV' in df.columns:
    pearson = df.corr(numeric_only=True)['MEDV'].sort_values(ascending=False)
    spearman = df.corr(numeric_only=True, method='spearman')['MEDV'].sort_values(ascending=False)

    corr_table = pd.DataFrame({'pearson': pearson, 'spearman': spearman})
    display(corr_table.round(3))

    print('Top 8 by |pearson| (excluding MEDV itself):')
    top_abs = pearson.drop(labels=['MEDV']).abs().sort_values(ascending=False).head(8)
    display(pd.DataFrame({'abs_pearson': top_abs, 'pearson': pearson[top_abs.index]}).round(3))

# Strongest predictor–predictor correlations (absolute, unique pairs)
predictors = [c for c in df.columns if c != 'MEDV']
cm = df[predictors].corr(numeric_only=True)
abs_vals = cm.abs().to_numpy(copy=True)
np.fill_diagonal(abs_vals, 0.0)
abs_cm = pd.DataFrame(abs_vals, index=cm.index, columns=cm.columns)

stacked = abs_cm.stack().sort_values(ascending=False)
seen = set()
pairs = []
for (a, b), abs_corr in stacked.items():
    key = tuple(sorted((str(a), str(b))))
    if key in seen:
        continue
    seen.add(key)
    pairs.append({'a': key[0], 'b': key[1], 'corr': float(cm.loc[a, b]), 'abs_corr': float(abs_corr)})
    if len(pairs) >= 15:
        break

display(pd.DataFrame(pairs).round(3))


## 5) More Results: Quantiles, Skewness, and the `MEDV=50` Cap

This makes the skew/outlier story more visible (e.g., `CRIM` heavy tail) and quantifies the well-known `MEDV` ceiling.

In [ ]:
# Quantiles + skewness
q = df[numeric_cols].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).T
q.columns = ['p01', 'p05', 'p50', 'p95', 'p99']
skew = df[numeric_cols].skew(numeric_only=True)

summary2 = q.join(skew.rename('skew'))
display(summary2.round(4))

# MEDV cap
if 'MEDV' in df.columns:
    cap_count = int((df['MEDV'] >= 50).sum())
    print('MEDV >= 50 count:', cap_count)
    display(df.loc[df['MEDV'] >= 50].head())

# A compact view of a few key variables
key_cols = [c for c in ['CRIM', 'RM', 'LSTAT', 'TAX', 'NOX', 'DIS', 'PTRATIO', 'RAD'] if c in df.columns]
display(summary2.loc[key_cols].round(4))


## 6) Visualizations (inline + saved to disk)

We generate several plots and save them into `outputs/boston_eda/` so you can reference them outside the notebook too.

In [ ]:
def savefig(name: str) -> Path:
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print('saved:', path)
    return path

# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap='vlag', center=0, square=True)
plt.title('Correlation heatmap')
plt.tight_layout()
savefig('corr_heatmap.png')
plt.show()

# MEDV distribution
if 'MEDV' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df['MEDV'], bins=30, kde=True)
    plt.title('MEDV distribution')
    plt.tight_layout()
    savefig('medv_hist.png')
    plt.show()

# Scatter + regression line for a few key predictors
if 'MEDV' in df.columns:
    for x in ['RM', 'LSTAT', 'PTRATIO', 'NOX']:
        if x not in df.columns:
            continue
        plt.figure(figsize=(6.5, 5))
        sns.scatterplot(data=df, x=x, y='MEDV', alpha=0.7, s=18)
        sns.regplot(data=df, x=x, y='MEDV', scatter=False, color='black', line_kws={'lw': 1})
        plt.title(f'MEDV vs {x}')
        plt.tight_layout()
        savefig(f'scatter_{x.lower()}_medv.png')
        plt.show()

# Boxplot by CHAS
if 'CHAS' in df.columns and 'MEDV' in df.columns:
    plt.figure(figsize=(6.5, 5))
    sns.boxplot(data=df, x='CHAS', y='MEDV')
    plt.title('MEDV by CHAS')
    plt.tight_layout()
    savefig('box_medv_by_chas.png')
    plt.show()


## 7) Persist Outputs: Save Tables + Summary JSON

This writes a few key tables into `outputs/boston_eda/` so results are easy to reuse in scripts/reports.

In [ ]:
# Save tables
(desc.round(6)).to_csv(OUTPUT_DIR / 'describe_numeric.csv')
print('saved:', OUTPUT_DIR / 'describe_numeric.csv')

if 'MEDV' in df.columns:
    pearson = df.corr(numeric_only=True)['MEDV'].sort_values(ascending=False)
    spearman = df.corr(numeric_only=True, method='spearman')['MEDV'].sort_values(ascending=False)
    pd.DataFrame({'pearson': pearson, 'spearman': spearman}).round(6).to_csv(OUTPUT_DIR / 'corr_with_medv.csv')
    print('saved:', OUTPUT_DIR / 'corr_with_medv.csv')

# Save a small JSON summary (paths + key facts)
summary = {
    'data_path': str(DATA_PATH),
    'shape': {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
    'missing_total': int(df.isna().sum().sum()),
    'duplicates': int(df.duplicated().sum()),
}
if 'MEDV' in df.columns:
    summary['medv_cap_at_50_count'] = int((df['MEDV'] >= 50).sum())

with open(OUTPUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('saved:', OUTPUT_DIR / 'summary.json')

print('\nArtifacts directory:', OUTPUT_DIR.resolve())


## 8) Export `.ipynb` → `.qmd` (Quarto convert) and Render

From the repo root, run:

```bash
quarto convert sandbox/boston_eda.ipynb --output sandbox/boston_eda.qmd
quarto render sandbox/boston_eda.qmd
```

If you have LaTeX installed and want PDF:

```bash
quarto render sandbox/boston_eda.qmd --to pdf
```

In [ ]:
# Path sanity check
print('cwd:', Path.cwd())
print('notebook:', Path('sandbox/boston_eda.ipynb').resolve())
print('data exists:', DATA_PATH.exists(), DATA_PATH.resolve())
print('output dir:', OUTPUT_DIR.resolve())
